<a href="https://colab.research.google.com/github/bradfa/colab-notebooks/blob/main/ministral3_3b_kv_cache_kld_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KV-Cache Quantization KLD Benchmark

Measures the impact of different KV-cache quantization approaches on
**Ministral-3-3B-Base-2512** using Kullback-Leibler Divergence (KLD),
perplexity ratio, and same-top-p metrics via llama.cpp's `llama-perplexity`.

## Design
- **Model**: Ministral-3-3B-Base-2512 — pure dense transformer, 26 layers,
  GQA 32 query / 8 KV heads, 131K vocab, Apache 2.0
- **Weights**: BF16 GGUF (unquantized weights; only the KV cache varies)
- **Corpus**: Wikitext-2 test set (chunks configurable; 128 on A100, 64 on T4)
- **Baseline**: BF16 KV cache → logit dump file (~10 GB on A100, ~5 GB on T4)
- **Sweep**: f16, q8_0, q5_1, q5_0, q4_1, q4_0, iq4_nl, plus asymmetric combos
- **Binary**: ai-dock/llama.cpp-cuda prebuilt with CUDA, fallback to source build

## Runtime estimates
| GPU  | Chunks | Approx total time |
|------|--------|-------------------|
| A100 | 128    | ~45-60 min        |
| T4   | 64     | ~1.5-2 hr         |

## Note on TurboQuant
TQ3/TQ4 (Zandieh et al., ICLR 2026) are not yet merged into mainline
llama.cpp and therefore unavailable in prebuilt binaries. Noted as future work.

## 0 — Configuration
Edit these if you want to customise the run.

In [ ]:
# ---- user-editable configuration ----

# set to None for auto-detection (128 on A100+, 64 on T4)
CHUNKS_OVERRIDE = None

# KV cache types to sweep  (each entry is a (key_type, value_type) tuple)
KV_CONFIGS = [
    ("f16",    "f16"),
    ("q8_0",   "q8_0"),
    ("q5_1",   "q5_1"),
    ("q5_0",   "q5_0"),
    ("q4_1",   "q4_1"),
    ("q4_0",   "q4_0"),
    ("iq4_nl", "iq4_nl"),
    # asymmetric: heavier K, lighter V
    ("q8_0",   "q4_0"),
    ("q5_0",   "q4_0"),
]

# baseline KV cache type (used for the logit dump)
BASELINE_K = "bf16"
BASELINE_V = "bf16"

# context size per chunk (llama.cpp default is 512; match for comparability)
CTX_SIZE = 512

# number of GPU layers to offload (-1 = all)
N_GPU_LAYERS = 99

# paths
WORK_DIR       = "/content/kv_bench"
MODEL_DIR      = f"{WORK_DIR}/model"
GGUF_PATH      = f"{WORK_DIR}/ministral-3-3b-base-bf16.gguf"
LOGITS_PATH    = f"{WORK_DIR}/baseline_logits.kld"
WIKITEXT_PATH  = f"{WORK_DIR}/wikitext-2-raw/wiki.test.raw"
LLAMA_CPP_DIR  = f"{WORK_DIR}/llama.cpp"
BIN_DIR        = f"{WORK_DIR}/bin"
RESULTS_CSV    = f"{WORK_DIR}/kld_results.csv"

## 1 — GPU Detection & Chunk Auto-Config

In [ ]:
import subprocess, os, re, shutil

os.makedirs(WORK_DIR, exist_ok=True)

def get_gpu_info():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"],
            text=True
        ).strip()
    except Exception:
        raise RuntimeError("No NVIDIA GPU detected. Enable GPU in Runtime > Change runtime type.")
    name, mem_mb = out.split(",")
    return name.strip(), int(mem_mb.strip())

GPU_NAME, GPU_MEM_MB = get_gpu_info()
IS_A100_CLASS = GPU_MEM_MB >= 35000

if CHUNKS_OVERRIDE is not None:
    N_CHUNKS = CHUNKS_OVERRIDE
elif IS_A100_CLASS:
    N_CHUNKS = 128
else:
    N_CHUNKS = 64

print(f"GPU          : {GPU_NAME}")
print(f"VRAM         : {GPU_MEM_MB} MB")
print(f"Chunks       : {N_CHUNKS}")
print(f"Tier         : {'A100-class' if IS_A100_CLASS else 'T4-class'}")

cuda_ver_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    text=True
).strip()
print(f"Driver       : {cuda_ver_out}")

nvcc_result = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
if nvcc_result.returncode == 0:
    nvcc_ver = nvcc_result.stdout
    cuda_match = re.search(r"release (\d+\.\d+)", nvcc_ver)
    CUDA_VER = cuda_match.group(1) if cuda_match else "unknown"
else:
    CUDA_VER = "unknown"
print(f"CUDA Toolkit : {CUDA_VER}")

## 2 — Install llama.cpp (prebuilt CUDA binary, fallback to source)

In [ ]:
import urllib.request, zipfile, glob, json

os.makedirs(BIN_DIR, exist_ok=True)

def try_prebuilt():
    """Attempt to download a prebuilt CUDA binary from ai-dock/llama.cpp-cuda."""
    print("Attempting ai-dock prebuilt CUDA binary...")
    try:
        api_url = "https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest"
        with urllib.request.urlopen(api_url) as resp:
            release = json.loads(resp.read().decode())

        cuda_major = CUDA_VER.split(".")[0] if CUDA_VER != "unknown" else "12"
        target_pattern = f"cuda-{cuda_major}"

        asset_url = None
        for asset in release.get("assets", []):
            name = asset["name"]
            if "linux" in name.lower() and target_pattern in name and name.endswith(".zip"):
                asset_url = asset["browser_download_url"]
                print(f"  Found asset: {name}")
                break

        if asset_url is None:
            print("  No matching prebuilt asset found.")
            return False

        zip_path = f"{WORK_DIR}/llama-prebuilt.zip"
        print(f"  Downloading {asset_url} ...")
        urllib.request.urlretrieve(asset_url, zip_path)

        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(BIN_DIR)
        os.remove(zip_path)

        candidates = glob.glob(f"{BIN_DIR}/**/llama-perplexity", recursive=True)
        if not candidates:
            print("  llama-perplexity not found in archive.")
            return False

        bin_subdir = os.path.dirname(candidates[0])
        if bin_subdir != BIN_DIR:
            for f in glob.glob(f"{bin_subdir}/*"):
                shutil.move(f, BIN_DIR)

        os.chmod(f"{BIN_DIR}/llama-perplexity", 0o755)
        print("  Prebuilt binary ready.")
        return True
    except Exception as e:
        print(f"  Prebuilt download failed: {e}")
        return False


def build_from_source():
    """Clone llama.cpp and build with CUDA support."""
    print("Building llama.cpp from source (this takes ~10-15 min on T4, ~5 min on A100)...")
    if not os.path.isdir(LLAMA_CPP_DIR):
        subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR],
            check=True
        )
    build_dir = f"{LLAMA_CPP_DIR}/build"
    subprocess.run(
        ["cmake", LLAMA_CPP_DIR, "-B", build_dir,
         "-DGGML_CUDA=ON", "-DBUILD_SHARED_LIBS=OFF",
         "-DCMAKE_BUILD_TYPE=Release"],
        check=True
    )
    n_jobs = os.cpu_count() or 4
    subprocess.run(
        ["cmake", "--build", build_dir, "--config", "Release",
         "-j", str(n_jobs), "--target", "llama-perplexity", "llama-quantize"],
        check=True
    )
    for binary in ["llama-perplexity", "llama-quantize"]:
        src = f"{build_dir}/bin/{binary}"
        if os.path.isfile(src):
            shutil.copy2(src, f"{BIN_DIR}/{binary}")
    for so in glob.glob(f"{build_dir}/bin/*.so") + glob.glob(f"{build_dir}/src/*.so"):
        shutil.copy2(so, BIN_DIR)
    print("  Source build complete.")


PERPLEXITY_BIN = f"{BIN_DIR}/llama-perplexity"

if not os.path.isfile(PERPLEXITY_BIN):
    if not try_prebuilt():
        build_from_source()

result = subprocess.run([PERPLEXITY_BIN, "--help"], capture_output=True, text=True)
if result.returncode not in (0, 1):
    raise RuntimeError(f"llama-perplexity not working: {result.stderr[:500]}")
print(f"\nllama-perplexity ready at {PERPLEXITY_BIN}")

## 3 — Download Model & Convert to BF16 GGUF

Ministral-3-3B-Base-2512 is distributed in FP8 safetensors format.
We convert to BF16 GGUF for unquantized-weight perplexity testing.

In [ ]:
if not os.path.isfile(GGUF_PATH):
    print("Downloading Ministral-3-3B-Base-2512 from Hugging Face...")
    subprocess.run(
        ["pip", "install", "-q", "huggingface_hub[hf_transfer]", "transformers", "sentencepiece", "protobuf"],
        check=True
    )

    from huggingface_hub import snapshot_download
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    snapshot_download(
        repo_id="mistralai/Ministral-3-3B-Base-2512",
        local_dir=MODEL_DIR,
        ignore_patterns=["*.md", "*.txt", "*.png", "*.jpg"],
    )
    print(f"Model downloaded to {MODEL_DIR}")

    if not os.path.isdir(LLAMA_CPP_DIR):
        subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR],
            check=True
        )
    subprocess.run(
        ["pip", "install", "-q", "-r", f"{LLAMA_CPP_DIR}/requirements.txt"],
        check=True
    )

    print("Converting to BF16 GGUF...")
    subprocess.run(
        ["python", f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py",
         MODEL_DIR,
         "--outtype", "bf16",
         "--outfile", GGUF_PATH],
        check=True
    )
    print(f"GGUF written to {GGUF_PATH}")
    print(f"Size: {os.path.getsize(GGUF_PATH) / 1e9:.2f} GB")
else:
    print(f"GGUF already exists: {GGUF_PATH}")
    print(f"Size: {os.path.getsize(GGUF_PATH) / 1e9:.2f} GB")

## 4 — Download Wikitext-2 Test Set

In [ ]:
if not os.path.isfile(WIKITEXT_PATH):
    print("Downloading Wikitext-2...")
    wt_zip = f"{WORK_DIR}/wikitext-2-raw-v1.zip"
    urllib.request.urlretrieve(
        "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-raw-v1.zip?ref=salesforce-research",
        wt_zip
    )
    with zipfile.ZipFile(wt_zip, 'r') as zf:
        zf.extractall(WORK_DIR)
    os.remove(wt_zip)
    print(f"Wikitext-2 test set: {WIKITEXT_PATH}")
    wt_size = os.path.getsize(WIKITEXT_PATH)
    print(f"Size: {wt_size / 1e6:.1f} MB")
else:
    print(f"Wikitext-2 already present: {WIKITEXT_PATH}")

## 5 — Baseline Logit Dump

Run perplexity with BF16 KV cache and save the full logit distributions
to a binary file. This is the reference for all KLD comparisons.

The logit file is large (~10 GB at 128 chunks with 131K vocab) but only
needs to exist on disk for the duration of the experiment. Each subsequent
KLD run reads it but does **not** produce additional large files.

In [ ]:
import time

def run_perplexity(cache_type_k, cache_type_v, kld_mode="dump", label=""):
    """Run llama-perplexity and return (stdout, stderr, elapsed_seconds).

    kld_mode:
      "dump"    -> baseline run, writes logits to LOGITS_PATH
      "compare" -> KLD run, reads LOGITS_PATH and computes divergence
      "none"    -> plain perplexity, no KLD
    """
    cmd = [
        PERPLEXITY_BIN,
        "-m", GGUF_PATH,
        "-f", WIKITEXT_PATH,
        "-ngl", str(N_GPU_LAYERS),
        "--chunks", str(N_CHUNKS),
        "-c", str(CTX_SIZE),
        "--flash-attn",
        "-ctk", cache_type_k,
        "-ctv", cache_type_v,
    ]
    if kld_mode == "dump":
        cmd += ["--kl-divergence-base", LOGITS_PATH]
    elif kld_mode == "compare":
        cmd += ["--kl-divergence-base", LOGITS_PATH, "--kl-divergence"]

    tag = label or f"ctk={cache_type_k} ctv={cache_type_v}"
    print(f"\n{'='*60}")
    print(f"Running: {tag}  (mode={kld_mode})")
    print(f"{'='*60}")

    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = BIN_DIR + ":" + env.get("LD_LIBRARY_PATH", "")

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f"WARNING: non-zero return code {proc.returncode}")
        print(f"stderr (last 1000 chars):\n{proc.stderr[-1000:]}")

    lines = proc.stdout.strip().split("\n")
    for line in lines[-30:]:
        print(line)

    print(f"\nElapsed: {elapsed:.1f}s")
    return proc.stdout, proc.stderr, elapsed


# --- run baseline ---
if os.path.isfile(LOGITS_PATH):
    print(f"Baseline logits already exist at {LOGITS_PATH}")
    print(f"Size: {os.path.getsize(LOGITS_PATH) / 1e9:.2f} GB")
    print("Delete this file to regenerate.")
else:
    stdout, stderr, elapsed = run_perplexity(
        BASELINE_K, BASELINE_V,
        kld_mode="dump",
        label=f"BASELINE ctk={BASELINE_K} ctv={BASELINE_V}"
    )
    if os.path.isfile(LOGITS_PATH):
        print(f"\nBaseline logits saved: {os.path.getsize(LOGITS_PATH) / 1e9:.2f} GB")
    else:
        print("\nERROR: logit file was not created. Check stderr above.")

## 6 — KLD Sweep

For each KV cache configuration, run `llama-perplexity --kl-divergence`
against the baseline logits and parse the output.

In [ ]:
import pandas as pd

def parse_kld_output(stdout):
    """Extract KLD metrics from llama-perplexity stdout."""
    metrics = {}
    for line in stdout.split("\n"):
        line = line.strip()
        if "kl divergence" in line.lower() and "mean" in line.lower():
            m = re.search(r"([\d.]+(?:e[+-]?\d+)?)\s*\+/-\s*([\d.]+(?:e[+-]?\d+)?)", line)
            if m:
                metrics["kld_mean"] = float(m.group(1))
                metrics["kld_stderr"] = float(m.group(2))
        if "ppl ratio" in line.lower() or "perplexity ratio" in line.lower():
            m = re.search(r"([\d.]+(?:e[+-]?\d+)?)", line)
            if m:
                metrics["ppl_ratio"] = float(m.group(1))
        if "same top p" in line.lower():
            m = re.search(r"([\d.]+)%?", line)
            if m:
                metrics["same_top_p_pct"] = float(m.group(1))
        if "mean" in line.lower() and "delta" in line.lower() and "p" in line.lower():
            m = re.search(r"([\d.]+(?:e[+-]?\d+)?)", line)
            if m:
                metrics["mean_delta_p"] = float(m.group(1))
        if line.startswith("Final estimate:") or ("perplexity" in line.lower() and "final" in line.lower()):
            m = re.search(r"([\d.]+)\s*\+/-", line)
            if m:
                metrics["ppl"] = float(m.group(1))
    return metrics


def parse_kld_output_fallback(stdout):
    """Fallback: grab floating-point values from summary lines."""
    metrics = {}
    lines = stdout.strip().split("\n")
    for line in lines[-20:]:
        if "kl divergence" in line.lower():
            nums = re.findall(r"[\d.]+(?:e[+-]?\d+)?", line)
            if len(nums) >= 1 and "kld_mean" not in metrics:
                metrics["kld_mean"] = float(nums[0])
                if len(nums) >= 2:
                    metrics["kld_stderr"] = float(nums[1])
        if "same top" in line.lower():
            nums = re.findall(r"[\d.]+", line)
            if nums:
                metrics["same_top_p_pct"] = float(nums[0])
    return metrics


results = []

for (ctk, ctv) in KV_CONFIGS:
    label = f"ctk={ctk} ctv={ctv}"
    stdout, stderr, elapsed = run_perplexity(ctk, ctv, kld_mode="compare", label=label)

    metrics = parse_kld_output(stdout)
    if not metrics:
        metrics = parse_kld_output_fallback(stdout)

    metrics["cache_type_k"] = ctk
    metrics["cache_type_v"] = ctv
    metrics["label"] = f"K:{ctk} V:{ctv}"
    metrics["elapsed_s"] = round(elapsed, 1)
    results.append(metrics)

    if "kld_mean" in metrics:
        print(f"  -> KLD mean: {metrics['kld_mean']:.6f}")
    else:
        print("  -> WARNING: could not parse KLD from output")

df = pd.DataFrame(results)
df.to_csv(RESULTS_CSV, index=False)
print(f"\nResults saved to {RESULTS_CSV}")
print(df.to_string(index=False))

## 7 — Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11})

df = pd.read_csv(RESULTS_CSV)

if "kld_mean" not in df.columns or df["kld_mean"].isna().all():
    print("No KLD data to plot. Check the raw output above for errors.")
else:
    df_sorted = df.sort_values("kld_mean", ascending=True).reset_index(drop=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        f"KV-Cache Quantization Impact — Ministral-3-3B-Base\n"
        f"Baseline: K={BASELINE_K} V={BASELINE_V} | {N_CHUNKS} chunks | {GPU_NAME}",
        fontsize=13, fontweight="bold"
    )

    colors = plt.cm.viridis_r([i / len(df_sorted) for i in range(len(df_sorted))])

    # --- KLD bar chart ---
    ax = axes[0]
    bars = ax.barh(df_sorted["label"], df_sorted["kld_mean"], color=colors)
    if "kld_stderr" in df_sorted.columns:
        ax.errorbar(
            df_sorted["kld_mean"], df_sorted["label"],
            xerr=df_sorted["kld_stderr"], fmt="none", color="black",
            capsize=3, linewidth=0.8
        )
    ax.set_xlabel("Mean KL Divergence (lower = closer to baseline)")
    ax.set_title("KL Divergence")
    ax.invert_yaxis()

    # --- PPL ratio ---
    ax = axes[1]
    if "ppl_ratio" in df_sorted.columns and not df_sorted["ppl_ratio"].isna().all():
        ax.barh(df_sorted["label"], df_sorted["ppl_ratio"], color=colors)
        ax.axvline(x=1.0, color="red", linestyle="--", alpha=0.6, label="ratio=1.0")
        ax.set_xlabel("PPL Ratio (quantized / baseline)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "PPL ratio\nnot available", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
    ax.set_title("Perplexity Ratio")
    ax.invert_yaxis()

    # --- same top p ---
    ax = axes[2]
    if "same_top_p_pct" in df_sorted.columns and not df_sorted["same_top_p_pct"].isna().all():
        ax.barh(df_sorted["label"], df_sorted["same_top_p_pct"], color=colors)
        ax.set_xlabel("Same Top Token (%)")
    else:
        ax.text(0.5, 0.5, "Same-top-p\nnot available", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)
    ax.set_title("Same Top Prediction")
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/kld_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to {WORK_DIR}/kld_results.png")

## 8 — Summary Table

In [ ]:
if "kld_mean" in df.columns:
    display_cols = [c for c in ["label", "kld_mean", "kld_stderr", "ppl_ratio",
                                 "same_top_p_pct", "elapsed_s"] if c in df_sorted.columns]
    summary = df_sorted[display_cols].copy()
    col_names = {
        "label": "Config", "kld_mean": "KLD Mean", "kld_stderr": "KLD StdErr",
        "ppl_ratio": "PPL Ratio", "same_top_p_pct": "Same Top %", "elapsed_s": "Time (s)"
    }
    summary.rename(columns=col_names, inplace=True)
    print(summary.to_markdown(index=False, floatfmt=".6f"))
else:
    print("No results to summarize.")

## 9 — Cleanup (optional)

The baseline logit file is large (~10 GB). Uncomment the cell below to delete it.

In [ ]:
# if os.path.isfile(LOGITS_PATH):
#     os.remove(LOGITS_PATH)
#     print(f"Deleted {LOGITS_PATH}")

## Notes

### Interpreting the results
- **KL Divergence (KLD)**: measures how much the quantized KV-cache shifts the
  output probability distribution vs the BF16 baseline. Lower is better.
  A value of 0 means identical distributions.
- **PPL Ratio**: ratio of quantized perplexity to baseline perplexity.
  Values close to 1.0 mean negligible quality loss.
- **Same Top %**: how often the most-probable token is the same between
  baseline and quantized. Higher is better.

### Key findings from prior work
- The K cache is generally more sensitive to quantization than the V cache.
- q8_0 KV cache typically shows negligible quality loss vs FP16/BF16.
- Asymmetric configs (e.g. K:q8_0 V:q4_0) can be a good compromise.
- Models with aggressive GQA (fewer KV heads) may be more sensitive.

### Planned extensions
- **Weight quantization interaction**: sweep KV cache types across Q8_0,
  Q6_K, Q5_K_M, Q4_K_M weight quants to find the Pareto frontier of
  total memory savings vs quality.
- **Larger models**: repeat on Ministral-3-8B and Ministral-3-14B to test
  whether the 3B results (expected pessimistic bound) hold at scale.
- **Devstral-Small-2 (24B)**: test with Q6_K weights on A100 40GB using
  both Wikitext-2 and a code-focused corpus (C or Go source) to see if
  KV cache quantization affects code differently than prose.
- **TurboQuant (TQ3/TQ4)**: once merged into mainline llama.cpp, add
  ("tq3", "tq3") and ("tq4", "tq4") to KV_CONFIGS.